# Human epithelial: annotation decisions

The decision table labels every cluster of `leiden_scVI_3.0` (Leiden 3.0) in `data/interim/latent_human_epithelial.h5ad` (July lineage fit);
a removed cluster is labelled `EXCLUDE`. This notebook shows the evidence for each decision with the numbers of
the review dot plot: marker detection per cluster, QC (genes, UMIs, mitochondrial reads, doublet scores, array of
origin) and the fixed per-nucleus checks of 29 Oct. It writes the step's review dot plot, cluster table, marker
statistics and its rows of the removed-cluster ledger to `OUT_DIR`.

Fixed flag rules, against the kept clusters of this step (workflow/scripts/review/annotation_review.py):

| flag | rule |
|---|---|
| mixed | >= 15% of nuclei express two lineage panels (>= 2 genes each) and >= 3x the kept median |
| other lineage | (lineage steps) a panel kept clusters rarely express (median < 15%) is in >= 40% of nuclei and >= 4x the kept median, or one of its genes is detected in >= 50% of nuclei against < 10% (and 5x less) in kept clusters |
| doublet score | mean DoubletDetection score >= 5 and >= 5x the kept clusters' median |
| doublets | >= 20% of nuclei called doublets (rare: called doublets were removed at QC) |
| low RNA | median detected genes < half the kept median |
| small | fewer than 20 nuclei |
| fits another label | another label's mean profile correlates better (r >= 0.85, ahead by >= 0.05) |

An EXCLUDE cluster is "supported" when it is mixed, of another lineage, doublet-like or low in RNA; otherwise the
kit calls it a reinstatement candidate or asks for a look. The recorded reason next to each removed cluster comes
from `REASONS_PATH`, its decision provenance from `STEPS_PATH`.


In [ ]:
# Paths are relative to REPO. The rule render_review_notebook (workflow/rules/99b_review_notebooks.smk) sets
# every parameter; these defaults let the template run from notebooks/review/.
REPO = "../.."
SPECIES = "human"
STEP = "epithelial"
OBJECT_PATH = "data/interim/latent_human_epithelial.h5ad"
CLUSTER_COLUMN = "leiden_scVI_3.0"
DECISIONS_PATH = "resources/annotations/human_epithelial.csv"
VOCABULARY_PATH = "resources/annotation_vocabulary.yaml"
REVIEW_VERSION = None    # version tag of the decision set (config deliverables.version)
PANELS_PATH = None       # resources/review/panels_<tag>.json, as the rule resolves it
REASONS_PATH = None      # resources/review/exclusion_reasons_<tag>.csv
STEPS_PATH = None        # resources/review/decision_steps_<tag>.csv
V8_RELEASE_PATH = "data/deliverables/2026_10_29b/Cevrim_Human_All_Cells_extended.h5ad"    # v8 extended all-cells object: where each nucleus was in v8
V8_DECISIONS_PATH = "resources/annotations/human_epithelial.csv"    # v8 table of the same clustering (None for a round-2 re-fit)
JULY_OBJECT_PATH = None    # July fit of a round-2 lineage (None otherwise)
OUT_DIR = "results/annotation_review"

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Image, display

repo = Path(REPO).resolve()
sys.path.insert(0, str(repo / "workflow" / "scripts" / "review"))
from review_ledger import NOT_RELEASED, counts_text, july_origin, read_obs, removed_cluster_rows, v8_status  # noqa: E402
from review_step import ATLAS_ORDER, draw_decision_umap, label_order_and_display, read_decisions, review_step  # noqa: E402

# The decision set's version and records have no defaults: the rule passes them (papermill -p), as must a run of
# the template alone.
unset = [name for name, value in {"REVIEW_VERSION": REVIEW_VERSION, "PANELS_PATH": PANELS_PATH,
                                  "REASONS_PATH": REASONS_PATH, "STEPS_PATH": STEPS_PATH}.items() if value is None]
if unset:
    raise ValueError(f"parameters not set: {unset}")

PENDING_MARK = "PI sign-off pending"   # how a decision table's note flags a label awaiting the PI
PENDING_COLOUR = "#1F4E79"             # dark blue, apart from the red of EXCLUDE and the amber of the checks

# Verdicts, reasons and evidence are sentences; the tables below show them whole.
pd.set_option("display.max_colwidth", None, "display.width", 250, "display.max_rows", 200)


def in_repo(path):
    """Absolute path of a repository-relative path; None (input not used by this step) stays None."""
    return None if path in (None, "", "None") else repo / path

## Decisions and inputs
Checks that every cluster has one allowed label and every removed cluster a recorded reason.

In [ ]:
adata = ad.read_h5ad(in_repo(OBJECT_PATH))
decisions = pd.read_csv(in_repo(DECISIONS_PATH), dtype=str, keep_default_na=False)
labels = read_decisions(decisions)
vocabulary = yaml.safe_load(in_repo(VOCABULARY_PATH).read_text())

# Every cluster has exactly one decision, and every label is one this step may use.
clusters = set(adata.obs[CLUSTER_COLUMN].astype(str))
repeated = decisions.cluster[decisions.cluster.duplicated()].tolist()
if clusters != set(labels) or repeated:
    raise ValueError(f"clusters without a decision: {sorted(clusters - set(labels))}; rows for absent clusters: "
                     f"{sorted(set(labels) - clusters)}; repeated rows: {repeated}")
if STEP == "atlas":
    allowed = set(ATLAS_ORDER[SPECIES])
else:
    allowed = {label for cell_type, subtypes in vocabulary["allowed"].items() for label in [cell_type, *(subtypes or [])]}
unknown = sorted(set(labels.values()) - allowed - {"EXCLUDE"})
if unknown:
    raise ValueError(f"labels not allowed at this step: {unknown}")

# One recorded reason per EXCLUDE cluster, and one provenance row for the step.
reasons = pd.read_csv(in_repo(REASONS_PATH), dtype=str, keep_default_na=False)
reasons = reasons[(reasons.species == SPECIES) & (reasons.step == STEP)].set_index("cluster")
excluded = sorted((c for c, label in labels.items() if label == "EXCLUDE"), key=int)
if set(reasons.index) != set(excluded):
    raise ValueError(f"EXCLUDE clusters without a reason: {sorted(set(excluded) - set(reasons.index))}; "
                     f"reasons for kept clusters: {sorted(set(reasons.index) - set(excluded))}")
steps = pd.read_csv(in_repo(STEPS_PATH), dtype=str, keep_default_na=False)
step_rows = steps[(steps.species == SPECIES) & (steps.step == STEP)]
if len(step_rows) != 1:
    raise ValueError(f"{len(step_rows)} rows for {SPECIES} {STEP} in {STEPS_PATH}")
step_record = step_rows.iloc[0]

notes = decisions["note"] if "note" in decisions else pd.Series("", index=decisions.index)
pending = decisions[notes.str.contains(PENDING_MARK, case=False, regex=False)]

print(f"{adata.n_obs:,} nuclei in {len(clusters)} clusters of {CLUSTER_COLUMN}; every cluster has one decision")
print(f"{len(excluded)} EXCLUDE clusters, each with a recorded reason")
print(f"Decided at {step_record.decided_at} ({step_record.approval_date}, {step_record.approver}): {step_record.decision_record}")
print("PI sign-off pending: " + (", ".join(f"cluster {r.cluster} ({r.label})" for r in pending.itertuples()) or "none"))

## Review dot plot
Rows: clusters grouped by label, removed clusters in the red EXCLUDE block. Amber: the fixed checks of each flagged or removed cluster. Red: the recorded exclusion reason and its source.

In [ ]:
order, display_names = label_order_and_display(SPECIES, STEP, labels, vocabulary)
reason_text = {cluster: f"{row.reason} [{row.source_tag}]" for cluster, row in reasons.iterrows()}
pending_text = {row.cluster: f"{PENDING_MARK}: {display_names.get(row.label, row.label)}" for row in pending.itertuples()}

# Footnotes: how the decisions were made, what changed against the v8 table of the same clustering, pending items.
footnotes = [f"Decisions: {step_record.decided_at} ({step_record.approval_date}, {step_record.approver}); "
             f"{step_record.decision_record}."]
v8_decisions = read_decisions(in_repo(V8_DECISIONS_PATH)) if in_repo(V8_DECISIONS_PATH) else None
if v8_decisions is not None:
    changed = [f"{c} {v8_decisions.get(c, 'absent')} -> {labels[c]}" for c in sorted(labels, key=int)
               if v8_decisions.get(c) != labels[c]]
    footnotes.append("Changes against the v8 table of this clustering: " + ("; ".join(changed) or "none") + ".")
if pending_text:
    footnotes.append(f"{PENDING_MARK}: " + "; ".join(f"cluster {c} ({labels[c]})" for c in pending_text) + ".")

step_name = "whole-tissue atlas" if STEP == "atlas" else (f"{STEP}, round 2" if in_repo(JULY_OBJECT_PATH) else STEP)
result = review_step(
    SPECIES,
    STEP,
    OBJECT_PATH,
    CLUSTER_COLUMN,
    decisions,
    in_repo(OUT_DIR),
    in_repo(VOCABULARY_PATH),
    in_repo(PANELS_PATH),
    subtitle=f"Clusters: {OBJECT_PATH} ({CLUSTER_COLUMN}); decisions ({REVIEW_VERSION}): {DECISIONS_PATH}; "
             f"reasons: {REASONS_PATH}",
    notes=footnotes,
    reasons=reason_text,
    reasons_header="exclusion reason [source]",
    extra_text=("PI sign-off", pending_text, PENDING_COLOUR) if pending_text else None,
    step_name=step_name,
    adata=adata,
)
plt.close(result["figure"])
table = result["table"]
if result["missing_genes"]:
    print("panel genes not in this object:", result["missing_genes"])
display(Image(result["paths"]["png"], width=1400))

## Removed clusters
This step's rows of the removed-cluster ledger: QC, kit verdict, final reason and where the nuclei were in v8.

In [ ]:
cluster = adata.obs[CLUSTER_COLUMN].astype(str).to_numpy()
status = v8_status(adata.obs, read_obs(in_repo(V8_RELEASE_PATH)))
origin = None
if in_repo(JULY_OBJECT_PATH):
    origin = july_origin(adata.obs, read_obs(in_repo(JULY_OBJECT_PATH)), CLUSTER_COLUMN)

if origin is not None:
    fit = "round 2 (lineage re-fit)"
else:
    fit = "July whole-tissue fit" if STEP == "atlas" else "July lineage fit"
step_info = dict(species=SPECIES, step=STEP, fit=fit, object=OBJECT_PATH, cluster_column=CLUSTER_COLUMN,
                 decisions_table=DECISIONS_PATH,
                 **step_record[["decided_at", "approval_date", "approver", "decision_record"]].to_dict())
ledger = removed_cluster_rows(table, result["qc"], cluster, status, step_info, reasons, origin, v8_decisions)
ledger_path = in_repo(OUT_DIR) / f"{SPECIES}_{STEP}_removed_clusters.csv"
ledger.to_csv(ledger_path, index=False)

kept_reference = table.iloc[0]
print(f"{len(ledger)} clusters removed ({ledger.n_nuclei.sum():,} of {adata.n_obs:,} nuclei). Kept-cluster references: "
      f"median genes {kept_reference.ref_kept_median_genes:,.0f}; two-lineage median {kept_reference.ref_kept_median_two_lineages:.1f}%; "
      f"doublet-score median {kept_reference.ref_kept_median_doublet:.2f}")
ledger[["cluster", "n_nuclei", "median_genes", "median_umis", "mean_pct_mito", "mean_doublet_score",
        "median_doublet_score", "pct_doublet_score_above_5", "pct_two_lineages", "largest_array", "largest_array_share",
        "kit_verdict", "final_reason", "reason_source", "origin_v8"]].round(2)

Recorded evidence behind each removal, and the v8 wording where a reason was reworded.

In [ ]:
ledger[["cluster", "kit_best_label", "kit_best_r", "evidence_summary", "evidence_source", "reason_v8"]].round(2)

## Flags on kept clusters

In [ ]:
flagged = table[~table.excluded & (table.verdict != "fine")]
print(f"{len(flagged)} of {int((~table.excluded).sum())} kept clusters flagged by the fixed rules")
flagged[["cluster", "label", "n_nuclei", "flags", "median_genes", "mean_doublet", "pct_two_lineages",
         "best_label", "best_r", "own_label_r", "neighbour_labels", "top_genes"]].round(2)

## Where each cluster's nuclei were in v8
v8 status: the nucleus's label in the v8 extended release (`(extended only)`: supplementary population), or `not released`.

In [ ]:
rows = []
for name in sorted(set(cluster), key=int):
    member = cluster == name
    row = {"cluster": name, "decision": labels[name], "n_nuclei": int(member.sum()),
           "released_in_v8": int((status[member] != NOT_RELEASED).sum()), "v8_status": counts_text(status[member])}
    if origin is not None:
        row["origin"] = counts_text(origin[member])
    rows.append(row)
v8_by_cluster = pd.DataFrame(rows)

is_removed = v8_by_cluster.decision.eq("EXCLUDE")
for group, members in [("kept", v8_by_cluster[~is_removed]), ("removed", v8_by_cluster[is_removed])]:
    if len(members):
        print(f"{group} clusters: {members.released_in_v8.sum():,} of {members.n_nuclei.sum():,} nuclei released in v8")
v8_by_cluster

## UMAP
The step's own embedding (`X_umap_scVI`).

In [ ]:
draw_decision_umap(adata, CLUSTER_COLUMN, labels, order, display_names)
plt.show()

## Label counts

In [ ]:
counts = table.groupby("label").agg(clusters=("cluster", "size"), nuclei=("n_nuclei", "sum"))
counts = counts.sort_values("nuclei", ascending=False)
counts["display name"] = [display_names.get(label, label) for label in counts.index]
counts.loc["total"] = [counts.clusters.sum(), counts.nuclei.sum(), ""]
counts

## Files written

In [ ]:
for name, path in {**result["paths"], "removed_clusters": ledger_path}.items():
    print(f"{name}: {Path(path).relative_to(repo)}")